# RAG Assignment: Baseline (Bi-Encoder) vs. Re-Ranking (Cross-Encoder)
**Document:** Digital Personal Data Protection Act, 2023 (India) — 21-page PDF

| | Pipeline 1 (Baseline) | Pipeline 2 (Re-Ranking) |
|---|---|---|
| Retrieval | Bi-Encoder cosine similarity → Top 10 | same Top 10 |
| Re-rank | — | Cross-Encoder scores 10 (query, chunk) pairs |
| Context to LLM | Top 3 | Top 3 after re-ranking |
| Output | Answer 1 | Answer 2 |

Both pipelines run over the **exact same chunks** and use the **identical prompt template**.

**Run order:** Runtime → Run all. Step 2 asks you to upload the PDF (and optionally `a.txt` with the API key).

## 1. Setup

In [1]:
# poppler (pdftotext) is normally preinstalled on Colab; the apt line is a safety net.
!apt-get -qq install -y poppler-utils > /dev/null
!pip -q install sentence-transformers google-genai pandas numpy

E: Failed to fetch http://security.ubuntu.com/ubuntu/pool/main/p/poppler/poppler-utils_24.02.0-1ubuntu9.9_amd64.deb  404  Not Found [IP: 185.125.190.83 80]
E: Unable to fetch some archives, maybe run apt-get update or try with --fix-missing?


In [2]:
import os, re, time, glob, json, getpass, textwrap
import numpy as np
import pandas as pd
from IPython.display import display
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder
from google import genai
from google.genai import types

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

# ---- Models ----
BI_ENCODER   = "sentence-transformers/multi-qa-MiniLM-L6-cos-v1"   # dense retriever (Bi-Encoder)
CROSS_ENCODER = "cross-encoder/ms-marco-MiniLM-L-6-v2"             # re-ranker (as suggested in the brief)
LLM_MODEL    = "gemini-3.8-flash"                                  # answer generator (verify this id is valid for your API key)
TOP_K_RETRIEVE, TOP_K_CONTEXT = 10, 3

### API key
The key is **never written into the notebook** (the repo will be public). Priority order:
(single cell below handles all four)
1. Colab Secret named `GEMINI_API_KEY` (key icon in the left sidebar — recommended)
2. Environment variable `GEMINI_API_KEY`
3. An uploaded `a.txt` (key on the line after `API Key`)
4. Paste it when prompted

In [4]:
import os, glob, getpass

def _to_str(x):
    """Coerce whatever the prompt returns (str or dict) into a clean string."""
    if isinstance(x, dict):
        for k in ("value", "text", "content", "input", "data"):
            if isinstance(x.get(k), str):
                return x[k].strip()
        # last resort: first string value in the dict
        for v in x.values():
            if isinstance(v, str):
                return v.strip()
        return ""
    return (x or "").strip()

def load_api_key():
    # 1) Colab secret
    try:
        from google.colab import userdata
        k = userdata.get("GEMINI_API_KEY")
        if k:
            return _to_str(k)
    except Exception:
        pass
    # 2) Environment variable
    if os.environ.get("GEMINI_API_KEY"):
        return _to_str(os.environ["GEMINI_API_KEY"])
    # 3) a.txt with "API Key" on one line and the key on the next
    if os.path.exists("a.txt"):
        lines = [l.strip() for l in open("a.txt") if l.strip()]
        if "API Key" in lines:
            idx = lines.index("API Key") + 1
            if idx < len(lines):
                return lines[idx]
    # 4) Interactive prompt, tolerant of dict returns
    key = _to_str(getpass.getpass("Paste Gemini API key: "))
    if not key:
        key = _to_str(input("getpass came back empty, paste Gemini API key (visible): "))
    if not key:
        raise ValueError("No API key provided.")
    return key
PDF_PATH = glob.glob("*Data*Protection*.pdf")[0]
API_KEY = load_api_key()

# Ensure no environment variable is secretly forcing Vertex AI
os.environ.pop("GOOGLE_GENAI_USE_VERTEXAI", None)

if "LLM_MODEL" not in globals():
    LLM_MODEL = "gemini-2.5-flash"

# 1. Try the standard Gemini Developer API (AI Studio) FIRST — works for AI Studio 'AQ.' and 'AIza' keys
client = genai.Client(api_key=API_KEY, vertexai=False)
backend = "Gemini API"

try:
    reply = client.models.generate_content(model=LLM_MODEL, contents="Reply with the word OK").text.strip()
    print(f"PDF: {PDF_PATH} | backend: {backend}")
    print("LLM check:", reply)
except Exception as first_err:
    # 2. If the key is strictly a Vertex Express key, fall back to vertexai=True
    try:
        client = genai.Client(api_key=API_KEY, vertexai=True)
        backend = "Vertex AI"
        reply = client.models.generate_content(model=LLM_MODEL, contents="Reply with the word OK").text.strip()
        print(f"PDF: {PDF_PATH} | backend: {backend}")
        print("LLM check:", reply)
    except Exception as second_err:
        raise RuntimeError(
            f"Both backends rejected the AQ. key.\n"
            f"- Gemini API error: {str(first_err)[:250]}\n"
            f"- Vertex AI error: {str(second_err)[:250]}"
        )

PDF:  Digital Personal Data Protection Act, 2023 (India).pdf | backend: Gemini API
LLM check: OK


## 2. Load & chunk the document
Chunking strategy: **one chunk per section** of the Act (the natural legal unit). Sections longer than 1100 characters
are split into overlapping windows (150-char overlap) cut at clause/sentence boundaries. Each chunk is prefixed with
`[Section N]` so the LLM can cite it. The Schedule (penalty table) is split into one chunk per row.
Running headers, page numbers, chapter titles and most single-line marginal side-notes are stripped as noise
(a few multi-line margin notes may still leak into the text; the effect on retrieval is small).

In [5]:
!apt-get update -qq
!apt-get install -y poppler-utils

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libpoppler-dev libpoppler-private-dev libpoppler134
Recommended packages:
  poppler-data
The following NEW packages will be installed:
  poppler-utils
The following packages will be upgraded:
  libpoppler-dev libpoppler-private-dev libpoppler134
3 upgraded, 1 newly installed, 0 to remove and 56 not upgraded.
Need to get 1,528 kB of archives.
After this operation, 735 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 libpoppler-private-dev amd64 24.02.0-1ubuntu9.10 [197 kB]
Get:2 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 libpoppler-dev amd64 24.02.0-1ubuntu9.10 [5,202 B]
Get:3 http

In [6]:
import re, subprocess

def extract_text(pdf_path):
    """Plain-order text extraction (poppler). Keeps the marginal side-notes out of the way better than -layout."""
    return subprocess.run(["pdftotext", pdf_path, "-"], capture_output=True, text=True, check=True).stdout

NOISE = [r"THE GAZETTE OF INDIA EXTRAORDINARY", r"\[PART II—", r"SEC\. \d+\]", r"\d{1,2}", r"\d+ of \d{4}\.", r"CHAPTER [IVX]+"]
NOISE_RE = re.compile(r"^(?:%s)$" % "|".join(NOISE))

# Single-line marginal side-notes that pdftotext interleaves into the body text.
# (Multi-line margin notes such as 'Establishment of / Board.' cannot be told apart from real text safely.)
MARGIN_NOTES = {"Definitions.", "Notice.", "Consent.", "Exemptions.", "Penalties."}

def clean_lines(text):
    out = []
    for ln in text.splitlines():
        ln = ln.strip()
        if not ln or NOISE_RE.match(ln) or ln in MARGIN_NOTES:
            continue
        if ln.isupper() and len(ln) < 70 and not ln.startswith("("):   # chapter titles such as 'PRELIMINARY'
            continue
        out.append(ln)
    return out

def split_sections(text):
    """Return list of (section_id, text). Sections are numbered 1..44 in order; the Schedule is split per row."""
    body_start = text.index("BE it enacted")
    sched_start = text.index("THE SCHEDULE")
    body, sched = text[body_start:sched_start], text[sched_start:]

    lines = clean_lines(body)
    sections, cur_id, cur, expected = [], None, [], 1
    for ln in lines:
        m = re.match(r"^(\d{1,2})\.\s", ln)
        if m and int(m.group(1)) == expected:          # accept only the next section number -> ignores list items
            if cur_id is not None:
                sections.append((str(cur_id), " ".join(cur)))
            cur_id, cur, expected = expected, [ln], expected + 1
        elif cur_id is not None:
            cur.append(ln)
    sections.append((str(cur_id), " ".join(cur)))

    # Schedule: rows are numbered "1." ... "7." on their own line
    sched = sched.split("————")[0]
    rows = re.split(r"\n(\d)\.\n", sched)
    for i in range(1, len(rows) - 1, 2):
        sections.append((f"Schedule-{rows[i]}", "Schedule penalty row " + rows[i] + ": " + " ".join(rows[i + 1].split())))
    return sections

def make_chunks(sections, max_chars=1100, overlap=150):
    """One chunk per section if it fits, otherwise sliding windows cut at sentence/clause boundaries.
    Every chunk is prefixed with its section id so it is self-describing."""
    chunks = []
    for sec, txt in sections:
        txt = re.sub(r"\s+", " ", txt).strip()
        label = f"[Section {sec}] " if not sec.startswith("Schedule") else "[Schedule] "
        if len(txt) <= max_chars:
            pieces = [txt]
        else:
            pieces, start = [], 0
            while start < len(txt):
                end = min(start + max_chars, len(txt))
                if end < len(txt):
                    cut = max(txt.rfind("; ", start, end), txt.rfind(". ", start, end), txt.rfind(":— ", start, end))
                    if cut > start + max_chars // 2:
                        end = cut + 1
                pieces.append(txt[start:end].strip())
                if end >= len(txt): break
                start = max(end - overlap, start + 1)
                sp = txt.find(" ", start)             # start the next window on a word boundary
                start = sp + 1 if sp != -1 else len(txt)
        for j, p in enumerate(pieces):
            chunks.append({"chunk_id": len(chunks), "section": sec, "part": j + 1, "text": label + p})
    return chunks


raw_text = extract_text(PDF_PATH)
sections = split_sections(raw_text)
chunks   = make_chunks(sections)
chunk_texts = [c["text"] for c in chunks]
print(f"{len(sections)} sections/rows -> {len(chunks)} chunks; longest = {max(map(len, chunk_texts))} chars")
pd.DataFrame(chunks).head(8)

51 sections/rows -> 92 chunks; longest = 1111 chars


,chunk_id,section,part,text
0,0,1,1,"[Section 1] 1. (1) This Act may be called the Digital Personal Data Protection Act, 2023. (2) It shall come into for..."
1,1,2,1,"[Section 2] 2. In this Act, unless the context otherwise requires,— (a) “Appellate Tribunal” means the Telecom Dispu..."
2,2,2,2,"[Section 2] of contact to enable a Data Principal to give, manage, review and withdraw her consent through an access..."
3,3,2,3,[Section 2] (l) “Data Protection Officer” means an individual appointed by the Significant Data Fiduciary under clau...
4,4,2,4,[Section 2] gain a financial advantage otherwise than by way of legitimate remuneration; (q) “Member” means a Member...
5,5,2,5,"[Section 2] that compromises the confidentiality, integrity or availability of personal data; (v) “prescribed” means..."
6,6,2,6,[Section 2] (z) “Significant Data Fiduciary” means any Data Fiduciary or class of Data Fiduciaries as may be notifie...
7,7,3,1,"[Section 3] 3. Subject to the provisions of this Act, it shall— (a) apply to the processing of digital personal data..."


## 3. Index the chunks (Bi-Encoder)

In [7]:
bi_encoder = SentenceTransformer(BI_ENCODER)
# normalize_embeddings=True -> unit vectors, so inner product == cosine similarity
doc_emb = bi_encoder.encode(chunk_texts, normalize_embeddings=True, show_progress_bar=True, batch_size=32)
print("Embedding matrix:", doc_emb.shape)

cross_encoder = CrossEncoder(CROSS_ENCODER)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

## 4. The two pipelines
`baseline_retrieve` → Top 10 by cosine similarity. `rerank` → scores those same 10 pairs with the Cross-Encoder and re-sorts.
Both are timed separately for the latency section.

In [8]:
def baseline_retrieve(query, k=TOP_K_RETRIEVE):
    """Pipeline 1 retrieval: embed the query, cosine-similarity against all chunk vectors, return Top-k."""
    t0 = time.perf_counter()
    q = bi_encoder.encode([query], normalize_embeddings=True)[0]
    sims = doc_emb @ q                          # cosine similarity with every chunk
    top = np.argsort(-sims)[:k]                 # indices of the k best chunks, best first
    dt = time.perf_counter() - t0
    return [(int(i), float(sims[i])) for i in top], dt

def rerank(query, candidates):
    """Pipeline 2: score each (query, chunk) pair jointly with the Cross-Encoder and re-sort descending."""
    t0 = time.perf_counter()
    pairs = [(query, chunk_texts[i]) for i, _ in candidates]
    scores = cross_encoder.predict(pairs)
    order = np.argsort(-scores)
    dt = time.perf_counter() - t0
    return [(candidates[j][0], float(scores[j])) for j in order], dt

PROMPT = """You are a legal assistant answering questions about the Digital Personal Data Protection Act, 2023 (India).
Answer ONLY using the context below. If the context does not contain the answer, say "Not found in the provided context."
Be concise and cite the section numbers you rely on.

Context:
{context}

Question: {question}

Answer:"""

def build_prompt(query, chunk_ids):
    ctx = "\n\n".join(chunk_texts[i] for i in chunk_ids)
    return PROMPT.format(context=ctx, question=query)

def ask_llm(prompt, retries=4):
    for a in range(retries):
        try:
            r = client.models.generate_content(model=LLM_MODEL, contents=prompt,
                                               config=types.GenerateContentConfig(temperature=0))
            return r.text.strip()
        except Exception as e:           # rate limit / transient server error -> back off and retry
            if a == retries - 1: return f"[LLM error: {e}]"
            time.sleep(5 * (a + 1))

# Optional chunk-level gold: query -> phrase that must appear in the chunk (a section can span many chunks).
GOLD_PHRASE = {}

def is_gold(query, ch, gold_sections):
    if ch["section"] not in gold_sections: return False
    ph = GOLD_PHRASE.get(query)
    return ph is None or ph.lower() in ch["text"].lower()

def run_query(query, gold_sections=None, generate=True, verbose=True):
    base, t_base = baseline_retrieve(query)
    rer,  t_rer  = rerank(query, base)
    base_ids = [i for i, _ in base]
    rer_ids  = [i for i, _ in rer]
    res = {"query": query, "baseline": base, "reranked": rer, "t_base": t_base, "t_rerank": t_rer}

    if gold_sections:                    # rank (1-based) of the best-placed gold chunk (section match, plus GOLD_PHRASE if defined), None if absent from top 10
        def best_rank(ids):
            r = [n + 1 for n, i in enumerate(ids) if is_gold(query, chunks[i], gold_sections)]
            return min(r) if r else None
        res["gold_rank_base"], res["gold_rank_rerank"] = best_rank(base_ids), best_rank(rer_ids)

    if generate:
        res["answer1"] = ask_llm(build_prompt(query, base_ids[:TOP_K_CONTEXT]))   # Answer 1: baseline Top 3
        res["answer2"] = ask_llm(build_prompt(query, rer_ids[:TOP_K_CONTEXT]))    # Answer 2: re-ranked Top 3
    if verbose: show(res)
    return res

def show(res):
    print("=" * 110); print("QUERY:", res["query"]); print("=" * 110)
    ce = dict(res["reranked"]); new_rank = {i: n + 1 for n, (i, _) in enumerate(res["reranked"])}
    rows = [{"base_rank": n + 1, "chunk_id": i, "section": chunks[i]["section"], "cosine": round(s, 3),
             "CE_score": round(ce[i], 2), "rerank_rank": new_rank[i], "text": chunks[i]["text"][:90]}
            for n, (i, s) in enumerate(res["baseline"])]
    display(pd.DataFrame(rows))
    if "gold_rank_base" in res:
        print(f"Gold section rank: baseline = {res['gold_rank_base']}  ->  re-ranked = {res['gold_rank_rerank']}")
    if "answer1" in res:
        print("\n--- Answer 1 (baseline Top-3) ---\n" + textwrap.fill(res["answer1"], 110, replace_whitespace=False))
        print("\n--- Answer 2 (re-ranked Top-3) ---\n" + textwrap.fill(res["answer2"], 110, replace_whitespace=False))

# Smoke test. NOTE: the 'child' definition (s.2(f)) is not in the bi-encoder Top 10, so the gold rank is None for BOTH
# pipelines. A cross-encoder can only re-order the 10 candidates; it cannot recover a chunk that retrieval missed.
_ = run_query("What is the definition of a child under the Act?", gold_sections={"2"})

QUERY: What is the definition of a child under the Act?


,base_rank,chunk_id,section,cosine,CE_score,rerank_rank,text
0,1,30,9,0.362,-7.92,6,[Section 9] of personal data of a child by such classes of Data Fiduciaries or for such pu
1,2,75,38,0.362,-8.87,8,[Section 38] 38. (1) The provisions of this Act shall be in addition to and not in derogat
2,3,54,25,0.352,-9.47,9,"[Section 25] 25. The Chairperson, Members, officers and employees of the Board shall be de"
3,4,29,9,0.332,-7.22,3,"[Section 9] 9. (1) The Data Fiduciary shall, before processing any personal data of a chil"
4,5,76,39,0.330,-7.83,4,[Section 39] 39. No civil court shall have the jurisdiction to entertain any suit or proce
5,6,7,3,0.327,-7.88,5,"[Section 3] 3. Subject to the provisions of this Act, it shall— (a) apply to the processin"
6,7,71,35,0.319,-8.48,7,"[Section 35] 35. No suit, prosecution or other legal proceedings shall lie against the Cen"
7,8,68,32,0.317,-5.81,1,[Section 32] bar on proceedings under the provisions of this Act as regards the contents o
8,9,60,28,0.315,-7.22,2,[Section 28] inquire into the affairs of any person for ascertaining whether such person i
9,10,21,7,0.313,-9.81,10,[Section 7] accordance with the policy issued by the Central Government or any law for the


Gold section rank: baseline = None  ->  re-ranked = None

--- Answer 1 (baseline Top-3) ---
Not found in the provided context.

--- Answer 2 (re-ranked Top-3) ---
Not found in the provided context.


## 5. Screening candidate stress-test queries
Below are candidate queries written against the Act, each with the section(s) that actually contain the answer (`gold`).
This cell runs retrieval + re-ranking only (no LLM calls) and shows the rank of the gold chunk before and after re-ranking.
**Look for rows where `useful` is True, i.e. the gold chunk is at baseline rank 4–10 (or missing) and ends up in the re-ranked Top 3.**

> The gold sections are my reading of the Act — check them against the PDF before you rely on them, and add/replace queries freely.

In [9]:
CANDIDATES = [
  # (failure type, query, gold sections)
  ("Exception / corner case",
   "Are startups exempt from any obligations under the Act?", {"17"}),
  ("Exception / corner case",
   "Can the Government exempt its own agencies from the Act for national security reasons?", {"17"}),
  ("Exception / corner case",
   "Is notice and consent required when personal data is processed only for research or statistical purposes?", {"17"}),
  ("Exception / corner case",
   "Does the Act apply to data that the individual herself made publicly available?", {"3"}),
  ("Negation / constraint",
   "What is a Data Fiduciary not allowed to do with the personal data of children?", {"9"}),
  ("Negation / constraint",
   "Can personal data be transferred to every country outside India?", {"16"}),
  ("Negation / constraint",
   "Does the Act apply to personal data processed by an individual for personal or domestic purposes?", {"3"}),
  ("Negation / constraint",
   "Must a Data Fiduciary always delete data once consent is withdrawn?", {"6", "8"}),
  ("Keyword trap / distractor",
   "How much money can be imposed as a penalty if a company fails to protect personal data with security safeguards?", {"Schedule-1"}),
  ("Keyword trap / distractor",
   "What is the fine for not telling the Board about a personal data breach?", {"Schedule-2"}),
  ("Keyword trap / distractor",
   "How long does someone have to challenge a decision of the Board?", {"29"}),
  ("Keyword trap / distractor",
   "Under what situations can a person's data be used without asking for their consent?", {"7"}),
  # Extra queries aimed at negation / lexical-overlap traps (results unknown until run; keep only what the screening shows)
  ("Negation / constraint",
   "In which situations is a Data Fiduciary NOT required to give notice to the Data Principal before processing?", {"7", "17"}),
  ("Negation / constraint",
   "Which Data Fiduciaries are not subject to the rules on notice and on rights of Data Principals?", {"17"}),
  ("Keyword trap / distractor",
   "Which penalty applies when a Data Principal herself fails to perform her duties, such as giving false particulars?", {"Schedule-5", "15"}),
  ("Keyword trap / distractor",
   "What is the maximum monetary punishment for a Data Fiduciary that mishandles a child's data?", {"Schedule-3"}),
  ("Other",
   "Can the Chairperson of the Board be removed from office?", {"21", "22"}),
  ("Other",
   "Who can give consent on behalf of a person with a disability?", {"9", "2"}),
  ("Other",
   "What rights does a Data Principal have over their data?", {"11", "12", "13", "14"}),
  ("Other",
   "What additional duties apply to a Significant Data Fiduciary?", {"10"}),
]

# Chunk-level gold: the phrase that must be present in the answer-bearing chunk
GOLD_PHRASE.update({
  "Are startups exempt from any obligations under the Act?": "startup",
  "Can the Government exempt its own agencies from the Act for national security reasons?": "security of the State",
  "Is notice and consent required when personal data is processed only for research or statistical purposes?": "research, archiving",
})

screen = []
for ftype, q, gold in CANDIDATES:
    r = run_query(q, gold_sections=gold, generate=False, verbose=False)
    gb, gr = r["gold_rank_base"], r["gold_rank_rerank"]
    promoted = (gb is None or gb > 3) and gr is not None and gr <= 3
    screen.append({"type": ftype, "query": q, "gold": sorted(gold), "base_rank": gb, "rerank_rank": gr,
                   "useful": promoted})
screen_df = pd.DataFrame(screen)
screen_df

,type,query,gold,base_rank,rerank_rank,useful
0,Exception / corner case,Are startups exempt from any obligations under the Act?,[17],1.0,1.0,False
1,Exception / corner case,Can the Government exempt its own agencies from the Act for national security reasons?,[17],NaN,NaN,False
2,Exception / corner case,Is notice and consent required when personal data is processed only for research or statistical purposes?,[17],8.0,3.0,True
3,Exception / corner case,Does the Act apply to data that the individual herself made publicly available?,[3],1.0,1.0,False
4,Negation / constraint,What is a Data Fiduciary not allowed to do with the personal data of children?,[9],1.0,1.0,False
5,Negation / constraint,Can personal data be transferred to every country outside India?,[16],1.0,1.0,False
6,Negation / constraint,Does the Act apply to personal data processed by an individual for personal or domestic purposes?,[3],1.0,1.0,False
7,Negation / constraint,Must a Data Fiduciary always delete data once consent is withdrawn?,"[6, 8]",1.0,1.0,False
8,Keyword trap / distractor,How much money can be imposed as a penalty if a company fails to protect personal data with security safeguards?,[Schedule-1],1.0,1.0,False
9,Keyword trap / distractor,What is the fine for not telling the Board about a personal data breach?,[Schedule-2],10.0,1.0,True


### 5b. Extra negation stress queries
The first screening showed the bi-encoder handles simple negation fine (gold at rank 1). These five queries are harder: the
answer is a *prohibition / immunity / invalidity* clause ("no suit shall lie", "shall be invalid", "shall not have the right")
that sits in a short chunk, and the query uses negative or constraint wording that overlaps with many generic chunks.
They are screened the same way as above (retrieval + re-ranking only, no LLM calls) and appended to `screen_df`, so any with
`useful = True` are picked up automatically by `CHOSEN` in section 6. Latency profiling still uses the original 20 `CANDIDATES`.

> I have not run these. Keep only the ones the screening shows really fail in the baseline (base_rank > 3 or None) and are fixed after re-ranking (rerank_rank <= 3).

In [10]:
EXTRA_NEGATION = [
  ("Negation / constraint",
   "Who cannot be sued or prosecuted for actions taken in good faith under the Act?", {"35"}),
  ("Negation / constraint",
   "Is a Data Principal not entitled to know which other Data Fiduciaries have received her personal data?", {"11"}),
  ("Negation / constraint",
   "Can a Data Principal go to a civil court instead of the Board for a data protection dispute?", {"39"}),
  ("Negation / constraint",
   "Is consent still valid if part of it goes against another law?", {"6"}),
  ("Negation / constraint",
   "What must a Data Principal not do when providing her personal data or making a complaint?", {"15"}),
]

# Chunk-level gold: the phrase that must be present in the answer-bearing chunk
GOLD_PHRASE.update({
  "Is a Data Principal not entitled to know which other Data Fiduciaries have received her personal data?": "authorised by law",
  "Is consent still valid if part of it goes against another law?": "invalid to the extent",
  "What must a Data Principal not do when providing her personal data or making a complaint?": "impersonate",
})

extra = []
for ftype, q, gold in EXTRA_NEGATION:
    r = run_query(q, gold_sections=gold, generate=False, verbose=False)
    gb, gr = r["gold_rank_base"], r["gold_rank_rerank"]
    promoted = (gb is None or gb > 3) and gr is not None and gr <= 3
    extra.append({"type": ftype, "query": q, "gold": sorted(gold), "base_rank": gb, "rerank_rank": gr,
                  "useful": promoted})
extra_df = pd.DataFrame(extra)
screen_df = pd.concat([screen_df, extra_df], ignore_index=True)   # CHOSEN in section 6 reads screen_df
extra_df

,type,query,gold,base_rank,rerank_rank,useful
0,Negation / constraint,Who cannot be sued or prosecuted for actions taken in good faith under the Act?,[35],1.0,1.0,False
1,Negation / constraint,Is a Data Principal not entitled to know which other Data Fiduciaries have received her personal data?,[11],NaN,NaN,False
2,Negation / constraint,Can a Data Principal go to a civil court instead of the Board for a data protection dispute?,[39],9.0,9.0,False
3,Negation / constraint,Is consent still valid if part of it goes against another law?,[6],1.0,1.0,False
4,Negation / constraint,What must a Data Principal not do when providing her personal data or making a complaint?,[15],2.0,1.0,False


## 6. Failure-mode analysis (with LLM answers)
Pick **at least two** queries marked `USEFUL` above, plus the document-specific failure types (keyword trap, exception,
negation, buried chunk at rank 4–10). Put them in `CHOSEN` — the cell runs both pipelines with the LLM and prints the
rank table and both answers. Compare Answer 1 and Answer 2 against the text of the gold section.

In [11]:
# By default: every candidate the screening flagged as promoted into the Top 3.
CHOSEN = [(row.query, set(row.gold)) for row in screen_df.itertuples() if row.useful]
# Or choose manually, e.g.:
# CHOSEN = [("Are startups exempt from any obligations under the Act?", {"17"})]
print(len(CHOSEN), "queries selected")

results = []
for q, gold in CHOSEN:
    results.append(run_query(q, gold_sections=gold))
    time.sleep(2)   # stay under free-tier rate limits

2 queries selected
QUERY: Is notice and consent required when personal data is processed only for research or statistical purposes?


,base_rank,chunk_id,section,cosine,CE_score,rerank_rank,text
0,1,18,6,0.630,3.36,2,[Section 6] shall be registered with the Board in such manner and subject to such technica
1,2,11,5,0.618,4.23,1,[Section 5] Y shall accompany or precede the request for the personal data with notice to
2,3,10,5,0.587,2.74,4,[Section 5] 5. (1) Every request made to a Data Principal under section 6 for consent shal
3,4,13,6,0.557,2.55,5,"[Section 6] 6. (1) The consent given by the Data Principal shall be free, specific, inform"
4,5,15,6,0.535,0.77,7,[Section 6] Data Fiduciary to respond to any communication from the Data Principal for the
5,6,77,40,0.535,-1.50,9,"[Section 40] 40. (1) The Central Government may, by notification, and subject to the condi"
6,7,8,3,0.517,-2.21,10,"[Section 3] while blogging her views, has publicly made available her personal data on soc"
7,8,43,17,0.513,3.35,3,[Section 17] on the date on which it falls due. Y may process the personal data of X for a
8,9,33,11,0.503,1.77,6,[Section 11] 11. (1) The Data Principal shall have the right to obtain from the Data Fiduc
9,10,30,9,0.498,-0.80,8,[Section 9] of personal data of a child by such classes of Data Fiduciaries or for such pu


Gold section rank: baseline = 8  ->  re-ranked = 3

--- Answer 1 (baseline Top-3) ---
Not found in the provided context.

--- Answer 2 (re-ranked Top-3) ---
No. Under **Section 17(2)(b)**, the provisions of the Act do not apply to the processing of personal data that
is necessary for research, archiving, or statistical purposes, provided the personal data is not used to take
any decision specific to a Data Principal and the processing is carried on in accordance with prescribed
standards. Therefore, notice and consent requirements do not apply in this case.
QUERY: What is the fine for not telling the Board about a personal data breach?


,base_rank,chunk_id,section,cosine,CE_score,rerank_rank,text
0,1,56,27,0.566,2.97,2,[Section 27] 27. (1) The Board shall exercise and perform the following powers and functio
1,2,61,28,0.560,-6.22,9,[Section 28] not prevent access to any premises or take into custody any equipment or any
2,3,25,8,0.543,1.13,3,[Section 8] appropriate technical and organisational measures to ensure effective observan
3,4,59,28,0.492,-8.12,10,"[Section 28] 28. (1) The Board shall function as an independent body and shall, as far as"
4,5,18,6,0.474,-2.98,6,[Section 6] shall be registered with the Board in such manner and subject to such technica
5,6,57,27,0.443,1.05,4,"[Section 27] by a Consent Manager of its obligations in relation to her personal data, to"
6,7,14,6,0.432,-3.04,7,[Section 6] this Act or the rules made thereunder or any other law for the time being in f
7,8,27,8,0.431,-5.92,8,"[Section 8] be deemed to no longer be served, if the Data Principal does not–– (a) approac"
8,9,69,33,0.431,1.04,5,[Section 33] 33. (1) If the Board determines on conclusion of an inquiry that breach of th
9,10,86,Schedule-2,0.428,3.94,1,[Schedule] Schedule penalty row 2: Breach in observing the obligation to give the Board or


Gold section rank: baseline = 10  ->  re-ranked = 1

--- Answer 1 (baseline Top-3) ---
Not found in the provided context.

--- Answer 2 (re-ranked Top-3) ---
The penalty for failing to notify the Board of a personal data breach under sub-section (6) of Section 8 may
extend to two hundred crore rupees [Schedule, penalty row 2; Section 8(6)].


In [13]:
def show_gold(section):
    """Print the ground-truth text so you can judge whether each answer is complete/correct."""
    for c in chunks:
        if c["section"] == section: print(c["text"], "\n")

show_gold("17")

[Section 17] 17. (1) The provisions of Chapter II, except sub-sections (1) and (5) of section 8, and those of Chapter III and section 16 shall not apply where— (a) the processing of personal data is necessary for enforcing any legal right or claim; (b) the processing of personal data by any court or tribunal or any other body in India which is entrusted by law with the performance of any judicial or quasi-judicial or regulatory or supervisory function, where such processing is necessary for the performance of such function; (c) personal data is processed in the interest of prevention, detection, investigation or prosecution of any offence or contravention of any law for the time being in force in India; (d) personal data of Data Principals not within the territory of India is processed pursuant to any contract entered into with any person outside the territory of India by any person based in India; 

[Section 17] territory of India is processed pursuant to any contract entered into wit

## 7. Latency profiling
Each timing is the mean over repeated runs (after a warm-up) across all candidate queries. Baseline = query embedding +
cosine search over all chunks. Re-ranking = Cross-Encoder over the 10 candidates. LLM time is excluded (identical for both).

In [14]:
REPEATS = 5
print('Hardware:', 'GPU ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only', '| torch', torch.__version__)
queries = [q for _, q, _ in CANDIDATES]
baseline_retrieve(queries[0]); rerank(queries[0], baseline_retrieve(queries[0])[0])   # warm-up

tb, tr = [], []
for q in queries:
    for _ in range(REPEATS):
        cands, a = baseline_retrieve(q)
        _, b = rerank(q, cands)
        tb.append(a); tr.append(b)
tb, tr = np.array(tb) * 1000, np.array(tr) * 1000      # milliseconds

lat = pd.DataFrame({
    "Stage": ["Pipeline 1: baseline retrieval", "Pipeline 2: + cross-encoder re-rank", "Pipeline 2 total"],
    "mean (ms)":   [tb.mean(), tr.mean(), (tb + tr).mean()],
    "std (ms)":    [tb.std(),  tr.std(),  (tb + tr).std()],
    "p95 (ms)":    [np.percentile(tb, 95), np.percentile(tr, 95), np.percentile(tb + tr, 95)],
}).round(2)
display(lat)
print(f"Re-ranking adds ~{tr.mean() / tb.mean():.1f}x the baseline retrieval time.")

Hardware: CPU only | torch 2.11.0+cpu


,Stage,mean (ms),std (ms),p95 (ms)
0,Pipeline 1: baseline retrieval,28.14,15.76,61.40
1,Pipeline 2: + cross-encoder re-rank,1747.35,846.15,3601.72
2,Pipeline 2 total,1775.49,854.13,3635.33


Re-ranking adds ~62.1x the baseline retrieval time.


## 8. Scaling experiment (supports the "why not Cross-Encoder over everything?" question)
Time the Cross-Encoder over **all** chunks versus the bi-encoder search, then extrapolate.

In [15]:
q = queries[0]
cross_encoder.predict([(q, t) for t in chunk_texts[:8]])      # warm-up
runs = []
for _ in range(3):                                            # mean of 3 runs
    t0 = time.perf_counter(); cross_encoder.predict([(q, t) for t in chunk_texts]); runs.append(time.perf_counter() - t0)
t_ce_all = float(np.mean(runs))
_, t_bi = baseline_retrieve(q)

print(f"Cross-Encoder over all {len(chunks)} chunks: {t_ce_all*1000:.0f} ms   |   bi-encoder search: {t_bi*1000:.1f} ms")
per_pair = t_ce_all / len(chunks)
for n in (1_000, 100_000, 1_000_000):
    print(f"  extrapolated Cross-Encoder over {n:>9,} chunks: ~{per_pair * n / 60:8.1f} min per query")
# Bi-Encoder side: document vectors are computed ONCE offline; a query costs 1 encode + a vector search (ANN indexes make it sub-linear).

Cross-Encoder over all 92 chunks: 11720 ms   |   bi-encoder search: 18.7 ms
  extrapolated Cross-Encoder over     1,000 chunks: ~     2.1 min per query
  extrapolated Cross-Encoder over   100,000 chunks: ~   212.3 min per query
  extrapolated Cross-Encoder over 1,000,000 chunks: ~  2123.2 min per query


## 9. Save results (for writing the report)

In [16]:
out = []
for r in results:
    out.append({"query": r["query"],
                "gold_rank_baseline": r.get("gold_rank_base"), "gold_rank_reranked": r.get("gold_rank_rerank"),
                "baseline_top3": [chunks[i]["section"] for i, _ in r["baseline"][:3]],
                "reranked_top3": [chunks[i]["section"] for i, _ in r["reranked"][:3]],
                "answer1": r["answer1"], "answer2": r["answer2"]})
pd.DataFrame(out).to_csv("failure_analysis_results.csv", index=False)
lat.to_csv("latency_results.csv", index=False)
screen_df.to_csv("screening_results.csv", index=False)
print("Saved failure_analysis_results.csv, latency_results.csv, screening_results.csv")

Saved failure_analysis_results.csv, latency_results.csv, screening_results.csv


## 10. Report checklist (write this part yourself, in your own words)
1. **Architecture comparison** — bi-encoder (query & chunk encoded independently, cosine) vs cross-encoder (query+chunk in one sequence, joint self-attention).
2. **Rank shifts + answers** — from section 6: "Chunk X moved from rank a → b"; did Answer 1 omit a clause / misstate a rule / say "Not found"?
3. **Latency** — table from section 7.
4. **Scaling** — section 8: cross-encoder cost is linear in corpus size *per query* and cannot precompute document representations.